# EEG Seizure Detection Pipeline
**CHB-MIT Scalp EEG Database**  
Full pipeline: install → config → parse → preprocess → spectrogram → dataset → model → train → evaluate

# Section 1 — Install & Import

In [5]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'librosa', 'soundfile'])
print('librosa installed successfully.')

librosa installed successfully.


In [6]:
import os, re, gc, warnings, random, time
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('dark_background')

import mne
mne.set_log_level('WARNING')

# ── ModuleNotFoundError: librosa ──
import librosa
import librosa.display

from scipy.signal import butter, filtfilt, iirnotch, welch
from skimage.transform import resize

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print(f'MNE     : {mne.__version__}')
print(f'librosa : {librosa.__version__}')

Device  : cpu
PyTorch : 2.10.0+cpu
MNE     : 1.11.0
librosa : 0.11.0


# Section 2 — Global Configuration

In [9]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATA_ROOT = Path('/kaggle/input/datasets/amudhans07/chbmit-eeg-full-physionet-archive/chb_mit_part1')
SPEC_DIR  = Path('/kaggle/working/specs')
MODEL_DIR = Path('/kaggle/working/models')
SPEC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Patients to include. Extend to all 24 patients when scaling.
CASES = ['chb01', 'chb02', 'chb03']

# ── Signal ────────────────────────────────────────────────────────────────
FS         = 256          # native sampling rate (Hz)
N_CHANNELS = 23
STANDARD_CHANNELS = [
    'FP1-F7','F7-T7','T7-P7','P7-O1',
    'FP1-F3','F3-C3','C3-P3','P3-O1',
    'FP2-F4','F4-C4','C4-P4','P4-O2',
    'FP2-F8','F8-T8','T8-P8','P8-O2',
    'FZ-CZ','CZ-PZ',
    'P7-T7','T7-FT9','FT9-FT10','FT10-T8','T8-P8'
]

# ── Frequency bands ───────────────────────────────────────────────────────
FREQ_BANDS = {
    'delta': (0.5,  4.0),
    'theta': (4.0,  8.0),
    'alpha': (8.0, 13.0),
    'beta' : (13.0, 30.0),
    'gamma': (30.0, 70.0),
}

# ── Filtering ─────────────────────────────────────────────────────────────
BANDPASS_LOW  = 0.5
BANDPASS_HIGH = 70.0
NOTCH_FREQ    = 60.0    # US power-line noise
FILTER_ORDER  = 4

# ── Windowing ─────────────────────────────────────────────────────────────
WINDOW_SEC    = 4
OVERLAP_SEC   = 2
WINDOW_SAMP   = WINDOW_SEC * FS            # 1024 samples
STEP_SAMP     = (WINDOW_SEC - OVERLAP_SEC) * FS  # 512 samples
PRE_ICTAL_SEC = 5   # label windows within 5 s before onset as ictal

# ── Spectrogram ───────────────────────────────────────────────────────────
N_MELS   = 64
NPERSEG  = 128
NOVERLAP = 64
FMIN     = 0.5
FMAX     = 70.0
IMG_SIZE = (64, 64)   # final spectrogram spatial size fed to CNN

# ── Memory guard ──────────────────────────────────────────────────────────
# Cap interictal windows per EDF to prevent RAM exhaustion on Kaggle
MAX_INTERICTAL_PER_FILE = 1000 # You can adjust this number based on your RAM limits

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
NUM_EPOCHS   = 50
LR           = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE     = 10       # early stopping patience (epochs)

print('Config loaded.')
print(f'  DATA_ROOT  : {DATA_ROOT}')
print(f'  Spec cache : {SPEC_DIR}')
print(f'  Cases      : {CASES}')
print(f'  Max non-sz windows per file: {MAX_INTERICTAL_PER_FILE}')

Config loaded.
  DATA_ROOT  : /kaggle/input/datasets/amudhans07/chbmit-eeg-full-physionet-archive/chb_mit_part1
  Spec cache : /kaggle/working/specs
  Cases      : ['chb01', 'chb02', 'chb03']
  Max non-sz windows per file: 1000


# Section 3 — Parse Summary Files

In [10]:
def parse_summary(case_dir: Path) -> pd.DataFrame:
    case    = case_dir.name                    # e.g. 'chb01'
    summary = case_dir / f'{case}-summary.txt'
    if not summary.exists():
        print(f'[WARN] No summary found: {summary}')
        return pd.DataFrame()

    records = []
    current_file = None

    for line in summary.read_text(errors='replace').splitlines():
        line = line.strip()
        m = re.match(r'File Name:\s+(\S+)', line)
        if m:
            current_file = case_dir / m.group(1)
            continue
        m = re.match(r'Seizure(?:\s+\d+)?\s+Start Time:\s+(\d+)\s+seconds', line)
        if m:
            sz_start = int(m.group(1))
            continue
        m = re.match(r'Seizure(?:\s+\d+)?\s+End Time:\s+(\d+)\s+seconds', line)
        if m and current_file is not None:
            sz_end = int(m.group(1))
            records.append({
                'case'      : case,
                'edf_file'  : str(current_file),
                'sz_start'  : sz_start,
                'sz_end'    : sz_end,
                'sz_dur_sec': sz_end - sz_start,
            })

    return pd.DataFrame(records)


all_seiz = pd.concat(
    [parse_summary(DATA_ROOT / case) for case in CASES],
    ignore_index=True
)

print(f'Seizure events found      : {len(all_seiz)}')
print(f'EDF files with seizures   : {all_seiz["edf_file"].nunique()}')
display(all_seiz.head())
display(all_seiz.groupby('case')[['sz_dur_sec']].agg(['count','sum','mean']).round(1))

Seizure events found      : 17
EDF files with seizures   : 17


,case,edf_file,sz_start,sz_end,sz_dur_sec
0,chb01,/kaggle/input/datasets/amudhans07/chbmit-eeg-f...,2996,3036,40
1,chb01,/kaggle/input/datasets/amudhans07/chbmit-eeg-f...,1467,1494,27
2,chb01,/kaggle/input/datasets/amudhans07/chbmit-eeg-f...,1732,1772,40
3,chb01,/kaggle/input/datasets/amudhans07/chbmit-eeg-f...,1015,1066,51
4,chb01,/kaggle/input/datasets/amudhans07/chbmit-eeg-f...,1720,1810,90


sz_dur_sec           
           count  sum  mean
case                       
chb01          7  442  63.1
chb02          3  172  57.3
chb03          7  402  57.4

# Section 4 — Signal Preprocessing Utilities

In [11]:
# ─── Filtering helpers ────────────────────────────────────────────────────

def butter_bandpass(lowcut, highcut, fs, order=4):
    nyq = fs / 2
    return butter(order, [lowcut/nyq, highcut/nyq], btype='band')

def apply_bandpass(data, lowcut=BANDPASS_LOW, highcut=BANDPASS_HIGH, fs=FS, order=FILTER_ORDER):
    """Apply a zero-phase Butterworth bandpass filter to (n_ch, n_samp)."""
    b, a = butter_bandpass(lowcut, highcut, fs, order)
    return filtfilt(b, a, data, axis=-1)

def apply_notch(data, freq=NOTCH_FREQ, fs=FS, Q=30):
    """60 Hz notch filter for power-line noise."""
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data, axis=-1)

# ─── Robust channel alignment ─────────────────────────────────────

def align_channels(raw: mne.io.BaseRaw, target_chs: list) -> np.ndarray:
    """
    Return (n_channels, n_samples) array aligned to `target_chs`.
    Missing channels are filled with zeros so every EDF has the same
    shape — crucial when patients have slightly different montages.
    Channel names are normalised (uppercase, strip spaces) for matching.
    """
    available = {ch.upper().replace(' ', '-'): ch for ch in raw.ch_names}
    data_map  = {k: raw.get_data(picks=v)[0] for k, v in available.items()}

    n_samp  = raw.n_times
    aligned = np.zeros((len(target_chs), n_samp), dtype=np.float32)
    for i, ch in enumerate(target_chs):
        key = ch.upper().replace(' ', '-')
        if key in data_map:
            aligned[i] = data_map[key].astype(np.float32)
        # else: leave as zeros — channel not present in this recording
    return aligned

# ─── Common Average Reference (CAR) ───────────────────────────────

def common_average_reference(data: np.ndarray) -> np.ndarray:
    """Subtract the mean across non-zero channels at every time step."""
    mask = (data != 0).any(axis=-1)   # channels that are actually present
    if mask.sum() == 0:
        return data
    mean = data[mask].mean(axis=0, keepdims=True)
    data[mask] -= mean
    return data

# ─── Per-channel z-score normalisation ─────────────────────────────

def zscore_normalize(data: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Per-channel z-score along the time axis."""
    mu  = data.mean(axis=-1, keepdims=True)
    std = data.std(axis=-1, keepdims=True) + eps
    return (data - mu) / std


def preprocess_raw(raw: mne.io.BaseRaw) -> np.ndarray:
    """
    Full preprocessing chain:
      EDF raw  →  align channels  →  bandpass  →  notch  →  CAR  →  z-score
    Returns (N_CHANNELS, n_samples) float32 array.
    """
    data = align_channels(raw, STANDARD_CHANNELS)  # (23, T)
    data = apply_bandpass(data)                     # 0.5–70 Hz
    data = apply_notch(data)                        # remove 60 Hz
    data = common_average_reference(data)           # CAR
    data = zscore_normalize(data)                   # per-channel z-score
    return data

print('Preprocessing utilities loaded.')

Preprocessing utilities loaded.


# Section 5 — Mel-Spectrogram Extraction

In [12]:
def eeg_window_to_spectrogram(window: np.ndarray,
                               fs: int  = FS,
                               n_mels: int  = N_MELS,
                               n_fft: int   = NPERSEG,
                               hop: int     = NOVERLAP,
                               fmin: float  = FMIN,
                               fmax: float  = FMAX,
                               img_size: tuple = IMG_SIZE) -> np.ndarray:
    n_ch  = window.shape[0]
    specs = []
    for ch in range(n_ch):
        S = librosa.feature.melspectrogram(
            y=window[ch].astype(np.float32),
            sr=fs, n_fft=n_fft, hop_length=hop,
            n_mels=n_mels, fmin=fmin, fmax=fmax
        )
        S_db = librosa.power_to_db(S, ref=np.max)  # (n_mels, T_frames)
        specs.append(S_db)

    # Stack to (n_ch, n_mels, T_frames), then average over channels
    arr = np.stack(specs, axis=0).mean(axis=0)      # (n_mels, T_frames)

    # Resize to IMG_SIZE
    arr = resize(arr, img_size, anti_aliasing=True).astype(np.float32)

    # Normalise to [0, 1]
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

    return arr[np.newaxis, ...]   # (1, H, W)


print('Spectrogram extractor loaded.')

Spectrogram extractor loaded.


# Section 6 — EDF Loading & Window Extraction

In [14]:
def load_edf_safe(edf_path: str) -> mne.io.BaseRaw | None:
    try:
        raw = mne.io.read_raw_edf(edf_path, preload=False, verbose=False)
        return raw
    except Exception as e:
        print(f'[WARN] Cannot read {edf_path}: {e}')
        return None


def extract_windows(edf_path: str,
                    seizure_events: pd.DataFrame,
                    spec_dir: Path,
                    case: str,
                    max_interictal: int = MAX_INTERICTAL_PER_FILE
                   ) -> list[dict]:
    raw = load_edf_safe(edf_path)
    if raw is None:
        return []

    # Resample if needed
    if round(raw.info['sfreq']) != FS:
        raw.resample(FS, npad='auto')

    # Load the entire signal into memory at once (float32 to save ~50% RAM
    # compared to float64). We'll discard it right after windowing.
    signal = preprocess_raw(raw)   # (23, T)  float32
    del raw; gc.collect()

    n_samp   = signal.shape[1]
    file_key = Path(edf_path).stem

    # Build seizure mask (sample-level) — much faster than per-window checks
    sz_mask = np.zeros(n_samp, dtype=bool)
    for _, ev in seizure_events.iterrows():
        s = int(ev['sz_start'] * FS)
        e = int(ev['sz_end']   * FS)
        # Optionally expand backwards by PRE_ICTAL_SEC
        s = max(0, s - PRE_ICTAL_SEC * FS)
        sz_mask[s : min(e, n_samp)] = True

    records     = []
    n_interictal = 0

    starts = range(0, n_samp - WINDOW_SAMP + 1, STEP_SAMP)
    for w_idx, start in enumerate(starts):
        end   = start + WINDOW_SAMP
        label = int(sz_mask[start:end].mean() >= 0.50)   # majority vote

        # Cap interictal windows to balance disk usage
        if label == 0:
            if n_interictal >= max_interictal:
                continue
            n_interictal += 1

        # Compute spectrogram and save to disk
        window = signal[:, start:end]                     # (23, 1024)
        spec   = eeg_window_to_spectrogram(window)        # (1, 64, 64)

        spec_path = spec_dir / case / f'{file_key}_w{w_idx:05d}_y{label}.npy'
        spec_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(spec_path, spec)

        records.append({'path': str(spec_path), 'label': label, 'case': case})

    del signal; gc.collect()
    return records


print('EDF loader / window extractor loaded.')

EDF loader / window extractor loaded.


# Section 7 — Build Spectrogram Cache & Manifest

In [ ]:
manifest_path = SPEC_DIR / 'manifest.csv'

if manifest_path.exists():
    print('Loading cached manifest...')
    manifest = pd.read_csv(manifest_path)
else:
    print('Building spectrogram cache')
    all_records = []
    for case in CASES:
        case_dir  = DATA_ROOT / case
        case_seiz = all_seiz[all_seiz['case'] == case]
        edf_files = sorted(case_dir.glob('*.edf'))
        print(f'  {case}: {len(edf_files)} EDFs, {len(case_seiz)} seizure events')

        for edf_f in edf_files:
            file_seiz = case_seiz[case_seiz['edf_file'] == str(edf_f)]
            recs = extract_windows(
                str(edf_f), file_seiz, SPEC_DIR, case,
                max_interictal=MAX_INTERICTAL_PER_FILE
            )
            all_records.extend(recs)
            gc.collect()

    manifest = pd.DataFrame(all_records)
    manifest.to_csv(manifest_path, index=False)
    print(f'Cache complete. Total windows: {len(manifest)}')

print(f'\nManifest: {len(manifest)} windows')
print(manifest['label'].value_counts().rename({0: 'Interictal', 1: 'Ictal'}))
print(f'Imbalance ratio: {(manifest["label"]==0).sum() / max((manifest["label"]==1).sum(),1):.1f}:1')

Building spectrogram cache
  chb01: 42 EDFs, 7 seizure events


# Section 8 — Train / Val / Test Split

In [ ]:
paths  = manifest['path'].values
labels = manifest['label'].values

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

print(f'Train : {len(y_tr):5d}  seizure={y_tr.mean():.3f}')
print(f'Val   : {len(y_val):5d}  seizure={y_val.mean():.3f}')
print(f'Test  : {len(y_te):5d}  seizure={y_te.mean():.3f}')

fig, ax = plt.subplots(figsize=(7, 3))
for i, (name, y) in enumerate({'Train': y_tr, 'Val': y_val, 'Test': y_te}.items()):
    ax.bar(i-0.2, (y==0).sum(), 0.35, color='#4e79a7', label='Non-Sz' if i==0 else '')
    ax.bar(i+0.2, (y==1).sum(), 0.35, color='#e15759', label='Seizure' if i==0 else '')
ax.set_xticks([0,1,2]); ax.set_xticklabels(['Train','Val','Test'])
ax.set_ylabel('Windows'); ax.set_title('Class counts per split'); ax.legend()
plt.tight_layout(); plt.show()

# Section 9 — Lazy Dataset & DataLoaders

In [ ]:
class LazyEEGDataset(Dataset):
    def __init__(self, paths, labels, augment=False):
        self.paths   = paths
        self.labels  = torch.from_numpy(labels).float()   # float for BCELoss
        self.augment = augment

    def __len__(self):  return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.load(self.paths[idx])).float()  # (1, H, W)
        if self.augment:
            x = self._spec_augment(x)
        return x, self.labels[idx]

    @staticmethod
    def _spec_augment(x, freq_p=8, time_p=8):
        # SpecAugment: random frequency and time masking.
        x = x.clone()
        _, H, W = x.shape
        f  = random.randint(0, freq_p);  f0 = random.randint(0, max(H-f, 0))
        x[:, f0:f0+f, :] = 0.0
        t  = random.randint(0, time_p);  t0 = random.randint(0, max(W-t, 0))
        x[:, :, t0:t0+t] = 0.0
        return x


def make_sampler(y):
    counts  = np.bincount(y.astype(int))
    weights = (1.0 / counts)[y.astype(int)]
    return WeightedRandomSampler(torch.DoubleTensor(weights), len(y), replacement=True)


train_ds = LazyEEGDataset(X_tr,  y_tr,  augment=True)
val_ds   = LazyEEGDataset(X_val, y_val, augment=False)
test_ds  = LazyEEGDataset(X_te,  y_te,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=make_sampler(y_tr),
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_loader))
print(f'Batch shape  : X={xb.shape}  y={yb.shape}')
print(f'Loaders      : train={len(train_loader)}  val={len(val_loader)}  test={len(test_loader)}')

# Section 10 — EEGConformer Architecture

In [ ]:
class EEGConformer(nn.Module):

    def __init__(self, num_classes=1, d_model=128, nhead=8, num_layers=4,
                 dropout=0.2):
        super().__init__()

        # ── CNN Backbone ───────────────────────────────────────────────
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(1,  32, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(32),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2),    # 64→32

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(64),
            nn.ELU(),
            nn.MaxPool2d(kernel_size=2, stride=2),    # 32→16

            # Block 3  — added for deeper representation
            nn.Conv2d(64, 128, kernel_size=3, padding=1, stride=1),
            nn.BatchNorm2d(128),
            nn.ELU(),
            # No pool here — preserve sequence length for Transformer
        )
        # After two pools: (B, 128, 16, 16)
        # Reshape to token sequence: (B, H=16, C*W = 128*16 = 2048)

        # ── Linear Projection of d_model ───────────────────────────────
        self.proj    = nn.Linear(128 * 16, d_model)   # 2048 → 128
        self.proj_drop = nn.Dropout(dropout)

        # ── Learnable Positional Encoding ─────────────────────────────
        # Shape (1, seq_len, d_model) — seq_len = H_out = 16
        self.pos_enc = nn.Parameter(torch.randn(1, 16, d_model) * 0.02)

        # ── Transformer Encoder ────────────────────────────────────────
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True           # Pre-LN is more stable to train
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # ── Classifier Head ────────────────────────────────────────────
        # No Sigmoid here — BCEWithLogitsLoss handles it internally
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.ELU(),
            nn.Linear(d_model // 2, 1)   # raw logit
        )

    def forward(self, x):
        # x: (B, 1, 64, 64)
        x = self.cnn(x)                                # (B, 128, 16, 16)

        B, C, H, W = x.shape
        x = x.view(B, H, C * W)                        # (B, 16, 2048)

        x = self.proj(x)                               # (B, 16, 128)
        x = self.proj_drop(x)
        x = x + self.pos_enc                           # positional encoding

        x = self.transformer(x)                        # (B, 16, 128)

        x = x.mean(dim=1)                              # global avg pooling (B, 128)
        return self.head(x)                            # (B, 1) raw logit


# Smoke-test the model
with torch.no_grad():
    dummy = torch.randn(4, 1, 64, 64)
    out   = EEGConformer()(dummy)
    print(f'Model output shape: {out.shape}  (expected: [4, 1])')

total_params = sum(p.numel() for p in EEGConformer().parameters())
print(f'Total parameters  : {total_params:,}')

# Section 11 — Training & Validation Loops

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()

    running_loss = 0.0
    all_logits   = []
    all_targets  = []

    for data, target in loader:
        # ── Move to device ────────────────────────────────────────────
        data   = data.to(device, non_blocking=True)          # (B, 1, H, W)
        target = target.to(device, non_blocking=True)        # (B,)

        # ── Zero gradients ────────────────────────────────────────────
        optimizer.zero_grad(set_to_none=True)   # faster than zero_grad()

        # ── Forward pass (AMP) ──────────────────────────
        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(data).squeeze(1)              # (B,)
                loss   = criterion(logits, target)
            scaler.scale(loss).backward()
            # Gradient clipping prevents exploding gradients in Transformer
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(data).squeeze(1)                  # (B,)
            loss   = criterion(logits, target)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        # ── Track metrics ─────────────────────────────────────────────
        running_loss += loss.item() * data.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(target.detach().cpu())

    # ── Epoch-level metrics ───────────────────────────────────────────
    avg_loss = running_loss / len(loader.dataset)
    logits_t = torch.cat(all_logits)
    tgts_t   = torch.cat(all_targets)
    probs    = torch.sigmoid(logits_t).numpy()
    tgts_np  = tgts_t.numpy().astype(int)

    auc = roc_auc_score(tgts_np, probs) if len(np.unique(tgts_np)) > 1 else 0.5
    f1  = f1_score(tgts_np, (probs >= 0.5).astype(int), zero_division=0)

    return {'loss': avg_loss, 'auc': auc, 'f1': f1}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_logits   = []
    all_targets  = []

    for data, target in loader:
        data   = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        logits = model(data).squeeze(1)              # (B,)
        loss   = criterion(logits, target)

        running_loss += loss.item() * data.size(0)
        all_logits.append(logits.cpu())
        all_targets.append(target.cpu())

    avg_loss = running_loss / len(loader.dataset)
    logits_t = torch.cat(all_logits)
    tgts_t   = torch.cat(all_targets)
    probs    = torch.sigmoid(logits_t).numpy()
    tgts_np  = tgts_t.numpy().astype(int)

    auc = roc_auc_score(tgts_np, probs) if len(np.unique(tgts_np)) > 1 else 0.5
    f1  = f1_score(tgts_np, (probs >= 0.5).astype(int), zero_division=0)

    return {'loss': avg_loss, 'auc': auc, 'f1': f1}


print('Training / validation loops loaded.')

# Section 12 — Main Training Pipeline

In [ ]:
def run_training_pipeline(
        model, train_loader, val_loader,
        epochs, device,
        lr=LR, weight_decay=WEIGHT_DECAY,
        patience=PATIENCE,
        save_path=MODEL_DIR / 'best_conformer.pt'
):
    model = model.to(device)

    # ── Optimiser ─────────────────────────────────────────────────────────
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    # ── Loss: BCEWithLogitsLoss + pos_weight for class imbalance ──────────
    n_neg     = int((y_tr == 0).sum())
    n_pos     = int((y_tr == 1).sum())
    pos_w     = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    print(f'pos_weight = {pos_w.item():.2f}  (neg:pos = {n_neg}:{n_pos})')

    # ── LR Scheduler: Cosine with warm restarts ───────────────────────────
    # T_0 = period of first restart (epochs); T_mult = factor to lengthen each period
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

    # ── Mixed-precision scaler (CUDA only) ────────────────────────────────
    scaler = torch.cuda.amp.GradScaler() if device.type == 'cuda' else None

    # ── Training state ────────────────────────────────────────────────────
    history       = defaultdict(list)
    best_val_auc  = 0.0
    patience_ctr  = 0

    print(f'\nStarting training: {epochs} epochs, device={device}')
    print(f'{"Epoch":>5} | {"Tr Loss":>8} | {"Tr AUC":>7} | {"Tr F1":>6} '
          f'| {"Va Loss":>8} | {"Va AUC":>7} | {"Va F1":>6} | {"LR":>9}')
    print('-' * 78)

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # ── Train ─────────────────────────────────────────────────────
        tr_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler)

        # ── Validate ──────────────────────────────────────────────────
        va_metrics = validate(model, val_loader, criterion, device)

        # ── Scheduler step ────────────────────────────────────────────
        scheduler.step(epoch - 1)     # CosineAnnealingWarmRestarts takes (epoch)
        current_lr = optimizer.param_groups[0]['lr']

        # ── Log ───────────────────────────────────────────────────────
        for k, v in tr_metrics.items(): history[f'train_{k}'].append(v)
        for k, v in va_metrics.items(): history[f'val_{k}'].append(v)
        history['lr'].append(current_lr)

        elapsed = time.time() - t0
        print(f'{epoch:>5} | {tr_metrics["loss"]:>8.4f} | {tr_metrics["auc"]:>7.4f} '
              f'| {tr_metrics["f1"]:>6.4f} | {va_metrics["loss"]:>8.4f} '
              f'| {va_metrics["auc"]:>7.4f} | {va_metrics["f1"]:>6.4f} '
              f'| {current_lr:>9.2e}  [{elapsed:.1f}s]')

        # ── Checkpoint ───────────────────────────────
        if va_metrics['auc'] > best_val_auc:
            best_val_auc = va_metrics['auc']
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'optim_state': optimizer.state_dict(),
                'val_auc'    : best_val_auc,
                'val_f1'     : va_metrics['f1'],
                'history'    : dict(history),
            }, save_path)
            patience_ctr = 0
            print(f'          ✓ New best val AUC: {best_val_auc:.4f} — model saved.')
        else:
            patience_ctr += 1

        # ── Early stopping ────────────────────────────────────────────
        if patience_ctr >= patience:
            print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs).')
            break

    print(f'\nTraining complete. Best val AUC: {best_val_auc:.4f}')
    return dict(history)


# ── Instantiate and run ────────────────────────────────────────────────────
model   = EEGConformer(d_model=128, nhead=8, num_layers=4, dropout=0.2)
history = run_training_pipeline(
    model, train_loader, val_loader,
    epochs=NUM_EPOCHS, device=DEVICE
)

# Section 13 — Training Curves & Final Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ── Plot training curves ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_range = range(1, len(history['train_loss']) + 1)

for ax, metric, title in zip(
    axes,
    ['loss', 'auc', 'f1'],
    ['Loss', 'ROC-AUC', 'F1-Score']
):
    ax.plot(epochs_range, history[f'train_{metric}'], label='Train', color='#4e79a7')
    ax.plot(epochs_range, history[f'val_{metric}'],   label='Val',   color='#e15759')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend()

plt.suptitle('EEGConformer Training History', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

# ── Load best checkpoint and evaluate on test set ──────────────────────────
ckpt = torch.load(MODEL_DIR / 'best_conformer.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best model from epoch {ckpt["epoch"]} (val AUC={ckpt["val_auc"]:.4f})')

test_metrics = validate(model, test_loader,
                        nn.BCEWithLogitsLoss(), DEVICE)
print(f'\nTest Results:')
print(f'  AUC  : {test_metrics["auc"]:.4f}')
print(f'  F1   : {test_metrics["f1"]:.4f}')
print(f'  Loss : {test_metrics["loss"]:.4f}')

# ── Confusion matrix ──────────────────────────────────────────────────────
@torch.no_grad()
def get_predictions(model, loader, device, threshold=0.5):
    model.eval()
    all_probs, all_tgts = [], []
    for data, target in loader:
        logits = model(data.to(device)).squeeze(1)
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_tgts.append(target.numpy())
    probs = np.concatenate(all_probs)
    tgts  = np.concatenate(all_tgts).astype(int)
    preds = (probs >= threshold).astype(int)
    return tgts, probs, preds

y_true, y_prob, y_pred = get_predictions(model, test_loader, DEVICE)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Interictal','Ictal'],
            yticklabels=['Interictal','Ictal'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout(); plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred,
                             target_names=['Interictal','Ictal']))

# ── ROC Curve ─────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, y_prob)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#e15759', lw=2,
        label=f'ROC AUC = {roc_auc_score(y_true, y_prob):.4f}')
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('Receiver Operating Characteristic')
ax.legend(); plt.tight_layout(); plt.show()